# 09 - Cross-Validate Against Global Crop Suitability 2024

GAEZ v5 has been the *only* source driving the crop recommendation so far - reasonable for a quick
labeling scheme, but not something to lean on alone for real analysis. Global Crop Suitability 2024
(Zenodo / Nature Scientific Data, https://zenodo.org/records/17666722) is a genuinely independent
suitability model - different authors, different methodology, newer baseline (~2024 vs GAEZ's
~2020) - covering 17 crops globally at ~1 km resolution as plain GeoTIFFs (verified: EPSG:4326,
43200x21600, float32, nodata=-1).

Of its 17 crops, 6 overlap with what this project now tracks via GAEZ: rice, maize, cassava,
soybean, oil palm, sugar cane. This notebook clips those 6 to Banten, zonal-stats them onto the
grid, and adds them as separate `gcs2024_<crop>` columns - **not** merged into the primary
labeling rule, so GAEZ's recommendation can be checked against an independent source rather than
taken on faith.

In [1]:
import os, sys
from pathlib import Path

NOTEBOOK_DIR = Path(os.path.abspath("."))
if NOTEBOOK_DIR.name != "preprocessing":
    NOTEBOOK_DIR = NOTEBOOK_DIR / "scripts" / "preprocessing"
os.chdir(NOTEBOOK_DIR)

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DOWNLOAD_DIR = PROJECT_ROOT / "scripts" / "download"
if str(DOWNLOAD_DIR) not in sys.path:
    sys.path.insert(0, str(DOWNLOAD_DIR))

from common import RAW_DIR, PROCESSED_DIR

print("cwd:", os.getcwd())

cwd: D:\Softwares\AnacondaProjects\GeoSuitability_AI_Analytics\scripts\preprocessing


## 1. Extract the 6 relevant crops from the archive (idempotent)

In [2]:
import zipfile

GCS_DIR = RAW_DIR / "global_crop_suitability_2024"
EXTRACT_DIR = GCS_DIR / "extracted"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# this project's crop name -> file name inside the archive
GCS_FILES = {
    "rice": "Rice.tif", "maize": "Maize.tif", "cassava": "Cassava.tif",
    "soybean": "Soybean.tif", "oilpalm": "Oilpalm.tif", "sugarcane": "Sugarcane.tif",
}

with zipfile.ZipFile(GCS_DIR / "2024.zip") as z:
    for crop, fname in GCS_FILES.items():
        dest = EXTRACT_DIR / fname
        if dest.exists():
            print(f"{crop}: already extracted, skipping")
            continue
        print(f"{crop}: extracting {fname}")
        z.extract(fname, EXTRACT_DIR)

print("Done")

rice: already extracted, skipping
maize: already extracted, skipping
cassava: already extracted, skipping
soybean: already extracted, skipping
oilpalm: already extracted, skipping
sugarcane: already extracted, skipping
Done


## 2. Clip each to the Banten extent

These are 43200x21600 global rasters (tiled, LZW-compressed). Each one is clipped with the `gdal_translate` CLI to
a small on-disk GeoTIFF, which is then read normally with rasterio, exactly like every other raster in this project.

Note: this route was first chosen because a direct rasterio windowed read (`.read(window=...)`) at the Banten
location kept killing the kernel. That turned out to be an environment problem, not GDAL: the notebook was being
executed from a shell where the conda environment was not activated, so its DLL folders (`Library/bin`) and
`GDAL_DATA` / `PROJ_LIB` were missing. In an activated environment (as PyCharm runs it) the windowed read works too;
the CLI route gives the same result and is kept.

In [3]:
import logging
import subprocess
import sys

import numpy as np
import rasterio

logging.getLogger("rasterio._env").setLevel(logging.ERROR)

from common import BANTEN_BBOX  # same bbox used by the download notebooks, with a touch of padding here

# gdal_translate isn't on PATH inside the kernel's subprocess environment - locate it relative to
# the active conda env's python.exe (standard conda-forge layout on Windows: Library/bin/*.exe).
GDAL_TRANSLATE = Path(sys.executable).parent / "Library" / "bin" / "gdal_translate.exe"
if not GDAL_TRANSLATE.exists():
    GDAL_TRANSLATE = "gdal_translate"  # fall back to PATH on non-Windows/other layouts

PAD = 0.15  # degrees
bbox = dict(
    west=BANTEN_BBOX["west"] - PAD, south=BANTEN_BBOX["south"] - PAD,
    east=BANTEN_BBOX["east"] + PAD, north=BANTEN_BBOX["north"] + PAD,
)

CLIP_DIR = GCS_DIR / "clipped"
CLIP_DIR.mkdir(parents=True, exist_ok=True)

clipped_paths = {}
for crop, fname in GCS_FILES.items():
    src_path = EXTRACT_DIR / fname
    dest_path = CLIP_DIR / f"{crop}_banten.tif"
    if not dest_path.exists():
        subprocess.run(
            [
                str(GDAL_TRANSLATE), "-q",
                "-projwin", str(bbox["west"]), str(bbox["north"]), str(bbox["east"]), str(bbox["south"]),
                str(src_path), str(dest_path),
            ],
            check=True,
        )
    clipped_paths[crop] = dest_path
    with rasterio.open(dest_path) as src:
        data = src.read(1)
        valid = data[data != src.nodata]
        print(
            f"{crop}: clipped shape={data.shape}, valid range=[{valid.min():.3f}, {valid.max():.3f}], "
            f"mean={valid.mean():.3f}"
        )

rice: clipped shape=(204, 240), valid range=[0.065, 0.995], mean=0.594
maize: clipped shape=(204, 240), valid range=[0.000, 0.740], mean=0.294
cassava: clipped shape=(204, 240), valid range=[0.286, 1.000], mean=0.833
soybean: clipped shape=(204, 240), valid range=[0.015, 0.729], mean=0.186
oilpalm: clipped shape=(204, 240), valid range=[0.023, 1.000], mean=0.618
sugarcane: clipped shape=(204, 240), valid range=[0.228, 0.978], mean=0.709


## Done for this notebook

The rest (zonal statistics, writing the `gcs2024_<crop>` columns, cross-validation against GAEZ) continues in
notebook 10; the 6 files clipped here are small (~200 KB each) and that is all it needs to read. The split was
first made because kernels kept dying after the heavy raster work; the cause was the non-activated conda environment
described above, not the amount of GDAL work, so the split is now just organisational.